In [1]:
using Plots,Distributions, JLD, DSP, Parameters, Match, SymDict, NumericalIntegration, Random, MAT, Pkg, CSV, DataFrames, XLSX, Pkg, JLD2, Colors, ColorSchemes, JLD2,  Makie, CairoMakie, Optim, Serialization, JuMP, Ipopt, Wandb, PyCall,Logging, GLM, PyPlot, Polynomials

folder_path = "./functions_run_simulations/"
for file in readdir(folder_path)
    if endswith(file, ".jl")
        include(joinpath(folder_path, file))
    end
end

┌ Warning: Error requiring `MakieCore` from `Polynomials`
│   exception = (LoadError("/Users/nmt121/.julia/packages/Polynomials/6i39P/ext/PolynomialsMakieCoreExt.jl", 1, ArgumentError("Package Polynomials does not have MakieCore in its dependencies:\n- If you have Polynomials checked out for development and have\n  added MakieCore as a dependency but haven't updated your primary\n  environment's manifest file, try `Pkg.resolve()`.\n- Otherwise you may need to report an issue with Polynomials")), Union{Ptr{Nothing}, Base.InterpreterIP}[Ptr{Nothing} @0x000000010ca6eb4e, Ptr{Nothing} @0x000000010ade09bf, Ptr{Nothing} @0x000000010ae11a0c, Ptr{Nothing} @0x000000010ae10e62, Ptr{Nothing} @0x000000010ae10a65, Ptr{Nothing} @0x000000010ae11104, Ptr{Nothing} @0x000000010ae11c34, Ptr{Nothing} @0x000000011ffb3d67, Ptr{Nothing} @0x000000010ade09bf, Ptr{Nothing} @0x000000011fea428a, Ptr{Nothing} @0x000000011fea3d62, Ptr{Nothing} @0x000000010ade09bf, Ptr{Nothing} @0x000000010aded259, Ptr{Nothing} @0x0

In [16]:
using JLD2, NPZ, Interpolations, Distributions

# List of days to process
days = [1,3,7,8,9,10,11,12,13,14]

# Load static neuron properties
JLD2.@load "Data_MN_Model/Data_final/Subjects_optim_process/VthCa_TA_subject1_gamma1.jld2" VthCa_newTA
nr_s = length(VthCa_newTA)

# Precompute neuron parameters (as before)
rheobase_S = LinRange(3.5, 6.5, nr_s) .+ rand(Normal(0.0, 0.05), nr_s)
rs_S = LinRange(38.75, 41.25, nr_s) ./ 10000
ls_S = LinRange(77.5, 82.5, nr_s) ./ 10000
rms_S = LinRange(1.15, 1.05, nr_s) .* 1000
rd_S = LinRange(20.75, 31.25, nr_s) ./ 10000
ld_S = LinRange(5500., 6800., nr_s) ./ 10000
rmd_S = LinRange(14.4, 10.7, nr_s) .* 1000

max_gna = 30.0
max_gkf_S = ones(nr_s) * 4
max_gks_S = LinRange(16, 25, nr_s)
max_gca_S = LinRange(0.038, 0.029, nr_s)

αm = ones(nr_s) * 22.0
βm = ones(nr_s) * 13.0
αh_S = ones(nr_s) * 0.5
βh_S = ones(nr_s) * 4
αn_S = ones(nr_s) * 1.5
βn_S = ones(nr_s) * 0.1
αq_S = ones(nr_s) * 1.5
βq_S = LinRange(0.025, 0.038, nr_s)
αp_D = 0.008
βp_D = LinRange(0.014, 0.016, nr_s)

Vthca_S = VthCa_newTA[1, :]
Apeak_S = LinRange(0.103, 0.123, nr_s)
tpeak_S = LinRange(110., 100., nr_s)
tetanic_S = LinRange(0.393, 0.491, nr_s)

γh = ones(nr_s)
IF = LinRange(1., 1., nr_s)

EL = 0; ENa = 120; ECa = 140; EK = -10; Ee = 70; Ei = -16
Ge = 0.6; Gi = 0.5
Vd0 = 0.; Vs0 = 0.;
rp = 5.0
pd=1; 

const step = 0.05
_int(num) = floor(Int, num)

# Precompute neuron properties
MN_list = []
for i in 1:nr_s
    gls = ((2*π*rs_S[i]*ls_S[i])/rms_S[i])*1e6
    gld = ((2*π*rd_S[i]*ld_S[i])/rmd_S[i])*1e6
    gc = ((2/(((70*ld_S[i])/(π*rd_S[i]^2)) + ((70*ls_S[i])/(π*rs_S[i]^2)))))*1e6
    rn = 1 / (gls + ((gld*gc)/(gld+gc)))
    Vth = round(rn * rheobase_S[i], digits=2)
    cs = 2*π*ls_S[i]*rs_S[i]*1*1e3
    cd = 2*π*ld_S[i]*rd_S[i]*1*1e3
    gna = 2*π*ls_S[i]*rs_S[i]*max_gna*1e3
    gkf = 2*π*ls_S[i]*rs_S[i]*max_gkf_S[i]*1e3
    gks = 2*π*ls_S[i]*rs_S[i]*max_gks_S[i]*1e3
    gca = 2*π*ld_S[i]*rd_S[i]*max_gca_S[i]*1e3

    S_MN = Dict(:Cs => cs, :Cd => cd, :Irh => rheobase_S[i], :gc => gc,
                 :gNa => gna, :gKs => gks, :gKf => gkf, :gCa => gca,
                 :gld => gld, :gls => gls, :αp => αp_D, :βp => βp_D[i],
                 :αq => αq_S[i], :βq => βq_S[i], :αm => αm[i], :βm => βm[i],
                 :αh => αh_S[i], :βh => βh_S[i], :αn => αn_S[i], :βn => βn_S[i],
                 :Vth => Vth, :VthCa => Vthca_S[i], :Rn => rn,
                 :Tetanic => tetanic_S[i], :Apeak => Apeak_S[i],
                 :IF => IF[i], :tpeak => tpeak_S[i], :γh => γh[i],
                 "type" => "All Type")
    push!(MN_list, S_MN)
end
MNs = MN_list

# Loop over days
for day in days
    day_str = lpad(day, 2, '0')  # Format day number with leading zero
    println("\nProcessing Day$day_str")

    # Load and resample force profile
    tr = npzread("/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Data_final/Force_exp/avg_forceS4_Day$(day_str).npy")
    sim_length = length(tr)
    interp = LinearInterpolation(1:sim_length, tr)
    resampled_indices = range(1, sim_length, length=Int(round(sim_length/2048))*1000)
    tr = interp(resampled_indices)
    sim_length = length(tr)
    amplitude = maximum(tr)

    # Simulation settings
    global tspan = 0:0.05:sim_length
    sim_time = _int(sim_length / 0.05)

    # Initialize state variables
    m=0.0; h=1.0; n=0.0; q=0.0; p=0.0; m0=0.0; h0=1.0; n0=0.0; q0=0.0; p0=0.0
    t0=-1.0; t1=-1.0; SPIKE0=0; SPIKE1=0
    global pool_state_params = fill((m, h, n, q, p, m0, h0, n0, q0, p0, t0, t1, SPIKE0, SPIKE1), nr_s)

    global spike_arr = zeros((nr_s, sim_time))
    global vs_arr = zeros((nr_s, sim_time))
    global pt_arr = zeros((nr_s, sim_time))
    global mt_arr = zeros((nr_s, sim_time))
    global ht_arr = zeros((nr_s, sim_time))
    global nt_arr = zeros((nr_s, sim_time))
    global qt_arr = zeros((nr_s, sim_time))

    global commoninput = tr
    global inh_input = fill(0.00001 * amplitude, sim_length)
    global full_input = zeros((nr_s, sim_length))
    global independantnoise = zeros((nr_s, sim_length))

    # Run simulation
    @time spike_array, vs_array, pt_arr = run_sim()

    # Save results using JLD2.save
    output_file = "Data_MN_Model/Data_final/Spikes/Force_analysis/DR_subject4_force$(day_str).jld2"
    JLD2.save(output_file, Dict("spike_array" => spike_array))
    println("Saved simulation results for Day$day_str to $output_file")
end



Processing Day01
1061.345004 seconds (42.04 G allocations: 686.758 GiB, 8.75% gc time, 0.02% compilation time)
Saved simulation results for Day01 to Data_MN_Model/Data_final/Spikes/Force_analysis/DR_subject4_force01.jld2

Processing Day03
149.027623 seconds (9.54 G allocations: 157.731 GiB, 10.70% gc time)
Saved simulation results for Day03 to Data_MN_Model/Data_final/Spikes/Force_analysis/DR_subject4_force03.jld2

Processing Day07
303.115304 seconds (18.51 G allocations: 306.868 GiB, 10.28% gc time)
Saved simulation results for Day07 to Data_MN_Model/Data_final/Spikes/Force_analysis/DR_subject4_force07.jld2

Processing Day08
272.563260 seconds (16.23 G allocations: 269.045 GiB, 10.74% gc time)
Saved simulation results for Day08 to Data_MN_Model/Data_final/Spikes/Force_analysis/DR_subject4_force08.jld2

Processing Day09
119.934092 seconds (7.20 G allocations: 118.958 GiB, 11.67% gc time)
Saved simulation results for Day09 to Data_MN_Model/Data_final/Spikes/Force_analysis/DR_subject4_f